In [1]:
from com.example.agentic.agent.LLMManager import LLMManager
# 
llm = LLMManager.create_llm('ollama')
llm.call('Hello')

'Hello again. Is there something I can help you with or would you like to chat?'

#### LOAD Documents

In [ ]:
from com.example.agentic.loader.LoadManager import LoadManager
# Website Data Ingestion 
documents = LoadManager.from_url(["https://docs.crewai.com/how-to/Installing-CrewAI/"])

#### Chunkings

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

#
print(len(chunks))


##### Embaddings

In [ ]:
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
from com.example.agentic.vectors.VectorStoreManager import VectorStoreManager

#chunks=[doc.page_content for doc in documents]

### Convert the text to embeddings
embedding_manager=EmbeddingManager()
embeddings = embedding_manager.embed_documents(documents)
print("[INFO] Example embedding:", embeddings[0] if len(embeddings) > 0 else None)

##store int the vector dtaabase
vectorstore = VectorStoreManager()
vectorstore.add_documents(documents,embeddings)

In [ ]:
import os
from crewai_tools import SerperDevTool,ScrapeWebsiteTool,TavilySearchTool

# Perform search for provide topic
websearch_tool = SerperDevTool()

#
scrape_tool = ScrapeWebsiteTool()

# 1. Initialize the tool with image search enabled
tavily_tool = TavilySearchTool(
    api_key= os.environ.get("TAVILY_API_KEY"),
    include_images=True,
    max_results=10
)

#### Agent Response Formatters

In [13]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict
from datetime import datetime

class ResearchPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key findings or insights about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Webpage information with title and url",
        default_factory=list
    )

class ResearchOutput(BaseModel):
    research_points: List[ResearchPoint] = Field(description="List of research findings")
    summary: str = Field(description="Brief summary of overall findings")

class ResearchImage(BaseModel):
    title: str = Field(description="The title of the image")
    url: str = Field(description="The url of the image")

class ResearchImageOutput(BaseModel):
    topic: str = Field(description="The details about primary topic")    
    images: List[ResearchImage] = Field(description="List of top images on topic")
    

class ExecutiveReportSection(BaseModel):
    section_emoji: str = Field(description="Section emoji (e.g., 🔍, 📊, 🎯)")
    section_title: str = Field(description="Section title")
    section_content: str = Field(description="Main content of the section")
    key_insights: List[str] = Field(description="Key insights from this section")
    recommendations: List[str] = Field(
        default_factory=list,
        description="Optional recommendations based on findings"
    )
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for this section",
        default_factory=list
    )

class ExecutiveReport(BaseModel):
    report_title: str = Field(description="Title of the report")
    generation_date: str = Field(description="Report generation date")
    executive_summary: str = Field(description="A concise executive summary")
    key_findings: List[Dict[str, str]] = Field(
        description="List of key findings with their sources",
        default_factory=list
    )
    report_sections: List[ExecutiveReportSection] = Field(
        description="Detailed report sections"
    )
    next_steps: List[str] = Field(description="Recommended next steps")
    sources: List[Dict[str, str]] = Field(
        description="All sources used in the report",
        default_factory=list
    )
    references : List[ResearchImage] = Field(description="References of top images on topic")

#### Crew Knowledge Base

In [3]:
#
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource
from crewai.knowledge.knowledge_config import KnowledgeConfig
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from com.example.agentic.tools.config import _embedder_config_ollama

# put in agent config knowledge_config=knowledge_config
knowledge_config = KnowledgeConfig(results_limit=10, score_threshold=0.5)

# Create custom storage with specific embedder
custom_storage = KnowledgeStorage(
    embedder=_embedder_config_ollama,
    collection_name="my_custom_knowledge"
)

text_kw_source = TextFileKnowledgeSource(
    file_paths=[
        "designs.txt", 
        "machine_learning.txt",
        "python_intro.txt"
    ]
)
# json_source.storage = custom_storage
#
from crewai.knowledge.source.json_knowledge_source import JSONKnowledgeSource
json_source = JSONKnowledgeSource(
    file_paths=[
        "designs-research.json"
    ]
)
#json_source.storage = custom_storage

In [14]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource
import os,requests,json
from datetime import datetime
from typing import Dict, Any
from pydantic import BaseModel, Field
from com.example.agentic.tools.config import _embedder_config_ollama

class ImagesKnowledgeSource(BaseKnowledgeSource):
    """Knowledge source that fetches relavent images url for design topic."""

    doc_path: str = Field(description="Json doc path")
    limit: int = Field(default=10, description="Number of images to fetch")

    def load_content(self) -> Dict[Any, str]:
        """Fetch and format desing images."""
        try:
            #response = requests.get(
            #    f"{self.api_endpoint}?limit={self.limit}"
            #)
            #response.raise_for_status()
            #data = response.json()

            work_dir = os.getenv("WORK_DIR")
            with open(f"{work_dir}/{self.doc_path}", 'r') as file:
                _json = json.load(file)
                
            _images = _json.get('design_reference_images', [])

            formatted_data = self.validate_content(_images)
            return {self.doc_path: formatted_data}
        except Exception as e:
            raise ValueError(f"Failed to fetch space news: {str(e)}")

    def validate_content(self, images: list) -> str:
        """Format articles into readable text."""
        formatted = "Design Reference Images:\n\n"
        for _image in images:
            formatted += f"""
                Title: {_image['title']}
                Url: {_image['url']}
                -------------------"""
        return formatted

    def add(self) -> None:
        """Process and store the articles."""
        content = self.load_content()
        for _, text in content.items():
            chunks = self._chunk_text(text)
            self.chunks.extend(chunks)

        self._save_documents()

    def aadd(self) -> None:
        pass
        

# Create knowledge source
desing_images = ImagesKnowledgeSource(
    doc_path="notebooks/knowledge/designs-research.json",
    limit=10
)

##### Design Crew

In [16]:
from datetime import datetime
from crewai import Task

from crewai_tools import ScrapeWebsiteTool, SerperDevTool
from com.example.agentic.tools.config import _embedder_config_ollama, _embedder_config_openai

@CrewBase
class DesignDevelopment():
    """Latest AI Design Development Crew"""

    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    @agent
    def content_researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['content_researcher'],
            verbose=True,
            tools=[
                SerperDevTool(),
                ScrapeWebsiteTool()
                # DesignSearchTool()
            ],
            #knowledge_sources=[text_kw_source],
            embedder=_embedder_config_openai,
            llm=llm
        )
    
    @agent
    def images_extractor(self) -> Agent:
        return Agent(
            config=self.agents_config['images_extractor'],
            knowledge_sources=[desing_images],
            verbose=True,
            tools=[],
            embedder=_embedder_config_openai,
            llm=llm
        )
    
    @agent
    def reporting_analyst(self) -> Agent:
        return Agent(
            config=self.agents_config['reporting_analyst'],
            verbose=True,
            llm=llm
        )
    
    @agent
    def formatter(self) -> Agent:
        return Agent(
            config=self.agents_config['formatter'],
            verbose=True,
            llm=llm
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'], 
            output_json=ResearchOutput
        )


    @task
    def extract_images_task(self) -> Task:
        return Task(
            config=self.tasks_config['extract_images_task'],
            output_json=ResearchImageOutput,
            #context=[self.research_task()]
        )
    
    
    @task
    def reporting_task(self) -> Task:
        return Task(
            config=self.tasks_config['reporting_task'],
            output_pydantic=ExecutiveReport
        )

    @task
    def formatting_task(self) -> Task:
        return Task(
            config=self.tasks_config['formatting_task']
        )

	
    @crew
    def crew(self) -> Crew:
        """Creates the DesignDevelopment Crew"""
        return Crew(
            agents=self.agents, # Automatically created by the @agent decorator
            tasks=self.tasks, # Automatically created by the @task decorator
            process=Process.sequential,
            verbose=True,
        )

In [17]:
from datetime import datetime
_exe_date = datetime.now().strftime('%Y-%m-%d')
_exe_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f" DesignDevelopment Crew triggered on {_exe_date} with execution id {_exe_id}")

def run():
    """
    Run the crew.
    """
    inputs = {
        'topic': 'Microservice Design',
        'date': _exe_date,
        'run_id': _exe_id
    }
    DesignDevelopment().crew().kickoff(inputs=inputs)

 DesignDevelopment Crew triggered on 2026-04-22 with execution id 20260422_114140


In [ ]:
run()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f901bd1e-21e7-490a-8f06-f276887283f7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: research_task                                                                                            │
│  ID: a3ae47ec-b6c9-43ed-aded-b8062873096a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Researcher                                                                                      │
│                                                                                                                 │
│  Task: Conduct a thorough research about Microservice Design Make sure you find minimume 10 relevant and        │
│  accurate content on solution design for Microservice Design.                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Researcher                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  research_points=[ResearchPoint(topic='Microservice Design', findings='Microservices are designed to be small,  │
│  independent applications that communicate with each other through APIs.', relevance='The rise of               │
│  microservices has led to a shift towards more flexibility and scalability in software development.',           │
│  sources=[{'title': 'What is Microservice Design?', 'url': 'https://en.wikipedia.org/wiki/Microservices'},      │
│  {'title': 'Benefits of Microservices', 'url': 'https://microservices.io/benefits'}]),                          │
│  ResearchPoint(topic='Microservice Communication', findings='Effective communication between microservices is   │
│  crucial for ensuring that they work together seamlessly.', relevance='The design of communication protocols    │
│  and APIs is essential to achieving this synergy.', sources=[{'title': 'Communicating with Microservices',      │
│  'url': 'https://devqa.com/microservices/communicate-microservices/'}, {'title': 'API Design for                │
│  Microservices', 'url': 'http://restfulpython.com/developingapis.html'}]), ResearchPoint(topic='Microservice    │
│  Scalability', findings='Scalability is a critical consideration when designing microservices architectures.',  │
│  relevance='The ability to scale horizontally or vertically is essential for handling increased load and        │
│  traffic.', sources=[{'title': 'Scaling Microservices', 'url':                                                  │
│  'https://www.dummies.com/how/technology/deployment/scaling-microservices.html'}, {'title': 'Containerization   │
│  for Scalability', 'url': 'https://blog.gavinbarnett.com/containerizing-scalsing-microservices/'}]),            │
│  ResearchPoint(topic='Microservice Caching', findings='Caching data between microservices can improve           │
│  performance and reduce latency.', relevance='The use of caching mechanisms can help resolve conflicts and      │
│  optimize resource utilization.', sources=[{'title': 'Implementing Cache for Microservices', 'url':             │
│  'https://www.tutorialspoint.com/microservices/cache_for_microservices.htm'}, {'title': 'Cache invalidation     │
│  Strategies', 'url': 'https://en.wikipedia.org/wiki/Cache_validity'}]), ResearchPoint(topic='Microservice       │
│  Architecture', findings='A microservices architecture consists of multiple smaller applications that           │
│  communicate with each other through APIs.', relevance='The design of the overall architecture is critical for  │
│  determining its scalability, maintainability, and performance.', sources=[{'title': 'Microservices             │
│  Architecture', 'url': 'https://microservices.io/architecture'}, {'title': 'Microservices Development Model',   │
│  'url': 'http://dzone.com/articles/microservices-development-model'}]), ResearchPoint(topic='Service            │
│  Discovery', findings='Service discovery is the process of finding and communicating with microservices in a    │
│  distributed system.', relevance='The use of service discovery mechanisms can help improve fault tolerance and  │
│  scalability.', sources=[{'title': 'Service Discovery Mechanisms', 'url':                                       │
│  'https://dzone.com/articles/service-discovery'}, {'title': 'Distributed Service Discovery', 'url':             │
│  'http://dataporalevel.org/ds/'}]), ResearchPoint(topic

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: research_task                                                                                            │
│  Agent: Content Researcher                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writing and Markdown Specialist                                                               │
│                                                                                                                 │
│  Task: Format the executive report into a beautiful markdown document without '```'. Follow these guidelines:   │
│  1. Use proper markdown headers (#, ##, ###) 2. Include emojis from the section_emoji field 3. Format key       │
│  findings and insights as bullet points 4. Add proper spacing and sections breaks 5. Make recommendations       │
│  stand out using blockquotes 6. Ensure the date is properly formatted 7. Add a table of contents at the         │
│  beginning 8. Use horizontal rules (---) to separate major sections 9. Add inline citations for each major      │
│  claim using (Author, Year) format 10. Include a Sources section at the end with numbered references 11. Each   │
│  source should include title, publisher/author, and URL 12. Link inline citations to the corresponding entry    │
│  in the Sources section 13. Ensure atleast one or more reference images include                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writing and Markdown Specialist                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Advanced Topics in Microservice Design Report - 2026-04-22                                                   │
│  =====================================================                                                          │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  This report provides an updated overview of the current state of microservice design. Key findings, best       │
│  practices, and source information are presented to assist developers and researchers.                          │
│                                                                                                                 │
│  ### Report Sections                                                                                            │
│                                                                                                                 │
│  #### Advantages of Containerization in Microservice Design                                                     │
│                                                                                                                 │
│  *   Benefits:                                                                                                  │
│      *   Containerization enables deployment, scaling, and service discovery for multiple microservices on a    │
│  single instance.                                                                                               │
│      *   Efficiency: containerization provides better resource utilization while maintaining high performance.  │
│  *   Implementing Docker or Kubernetes with image libraries can help leverage existing known pre-existing       │
│  libraries.                                                                                                     │
│                                                                                                                 │
│  #### Microservice Scalability                                                                                  │
│                                                                                                                 │
│  *   Benefits:                                                                                                  │
│      *   Containerization enables deployment, scaling, and service discovery for multiple microservices on a    │
│  single instance.                                                                                               │
│      *   Fault tolerance: containerization minimizes the need to restart instances.                             │
│  *   Consider implementing Docker or Kubernetes for efficient system management.                                │
│                                                                                                                 │
│  #### Microservice Caching                                                                                      │
│                                                                                                                 │
│  *   Benefits:                                         

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: formatting_task                                                                                          │
│  Agent: Technical Writing and Markdown Specialist                                                               │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f901bd1e-21e7-490a-8f06-f276887283f7                                                                       │
│  Final Output: # Advanced Topics in Microservice Design Report - 2026-04-22                                     │
│  =====================================================                                                          │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  This report provides an updated overview of the current state of microservice design. Key findings, best       │
│  practices, and source information are presented to assist developers and researchers.                          │
│                                                                                                                 │
│  ### Report Sections                                                                                            │
│                                                                                                                 │
│  #### Advantages of Containerization in Microservice Design                                                     │
│                                                                                                                 │
│  *   Benefits:                                                                                                  │
│      *   Containerization enables deployment, scaling, and service discovery for multiple microservices on a    │
│  single instance.                                                                                               │
│      *   Efficiency: containerization provides better resource utilization while maintaining high performance.  │
│  *   Implementing Docker or Kubernetes with image libraries can help leverage existing known pre-existing       │
│  libraries.                                                                                                     │
│                                                                                                                 │
│  #### Microservice Scalability                                                                                  │
│                                                                                                                 │
│  *   Benefits:                                                                                                  │
│      *   Containerization enables deployment, scaling, and service discovery for multiple microservices on a    │
│  single instance.                                                                                               │
│      *   Fault tolerance: containerization minimizes the need to restart instances.                             │
│  *   Consider implementing Docker or Kubernetes for efficient system management.                                │
│                                                                                                                 │
│  #### Microservice Caching                                                                                      │
│                                                                                                                 │
│  *   Benefits:                                        

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 2a355e7a-585a-4c71-b2ad-49ed611baa30                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/2a355e7a-585a-4c7 │
│ 1-b2ad-49ed611baa30?access_code=TRACE-8703a3d0d7                             │
│ 🔑 Access Code: TRACE-8703a3d0d7                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


In [ ]:
from crewai.tools import tool

@tool("Basic Functionality")
def my_simple_tool(question: str) -> str:
    """Tool to enhance the question provided."""
    tool_output = { "enhanced_question": "ef'{question} with clarifications'}
    return tool_output

In [ ]:
from crewai import Agent, Task, Crew, Process

from pydantic import BaseModel, Field

# Define a Pydantic model to validate order data
class OrderInputModel(BaseModel):
    order_id: int
    product_name: str
    quantity: int = Field(gt=0, description="Quantity must be greater than zero")

# Define a Pydantic model for the output structure
class OrderAnalysisOutputModel(BaseModel):
    total_orders: int
    top_product: str
    total_quantity: int

# Create an agent to analyze orders
order_analyst = Agent(
    role='Order Analyst',
    goal='Analyze the list of orders and provide a summary report.',
    backstory="You are an expert in processing and analyzing order data.",
    verbose=True,
    memory=True
)

# Define the task using the Pydantic models for input and output validation
order_analysis_task = Task(
    description="Analyze the incoming list of orders and provide a report on total orders, top product, and total quantity.",
    input_model=OrderInputModel,  # Pydantic model for input validation
    output_model=OrderAnalysisOutputModel,  # Pydantic model for output structure
    agent=order_analyst
)

# Define the crew and process
order_analysis_crew = Crew(
    agents=[order_analyst],
    tasks=[order_analysis_task],
    process=Process.sequential
)

# Sample input data (list of orders)
input_data = [
    {"order_id": 1, "product_name": "Laptop", "quantity": 2},
    {"order_id": 2, "product_name": "Monitor", "quantity": 5},
    {"order_id": 3, "product_name": "Laptop", "quantity": 3}
]

# Kick off the task
result = order_analysis_crew.kickoff(inputs={'input_data': input_data})

# The result will be validated and structured by the output_model
print(result)

####
####